# 第 6 章：如果网络不止一层，误差应该如何传回前面的层？

这一章的主线是反向传播：先完成前向传播得到预测，再用链式法则把最终误差逐层传回每个权重。

本 Notebook 按四步扩展网络：

1. 一个输入、两个权重；
2. 一个输入、两个隐藏节点；
3. 多个输入特征；
4. 多个样本同时训练。

每次扩展都只增加一种复杂度，同时追踪变量的形状。

## 6.1 最小的两层网络

先从只有一个数的网络开始：

```text
输入 x=2
  ↓ 乘 w1
隐藏值 h
  ↓ 乘 w2
预测值 prediction
```

前向传播是：`h = x * w1`，`prediction = h * w2`。

形状都是标量，所以这里看不出矩阵运算的复杂性；但误差如何穿过中间变量，已经是之后所有深层网络的同一个问题。

### 先做一次前向传播

假设：

```text
x = 2
w1 = 0.5
w2 = 0.4
target = 1.0
```

前向传播：
\[
h=2\times0.5=1
\]
\[
\hat{y}=1\times0.4=0.4
\]
平方误差：
\[
E=(0.4-1.0)^2=0.36
\]
现在问题来了：预测值太低，`w1` 和 `w2` 应该如何调整？

`w2` 比较容易理解：它直接影响预测值。

但 `w1` 没有直接连接预测值，中间隔着隐藏值 `h`。所以它的影响路径是：

```text
w1 → h → prediction → error
```
这就是链式法则：
\[
\frac{\partial E}{\partial w_1}
=
\frac{\partial E}{\partial \hat{y}}
\cdot
\frac{\partial \hat{y}}{\partial h}
\cdot
\frac{\partial h}{\partial w_1}
\]
分别计算：
\[
\frac{\partial E}{\partial \hat{y}}
=
2(\hat{y}-target)
\]
\[
\frac{\partial \hat{y}}{\partial h}=w_2
\]
\[
\frac{\partial h}{\partial w_1}=x
\]
所以：
\[
\frac{\partial E}{\partial w_1}
=
2(\hat{y}-target)\cdot w_2\cdot x
\]
代入：
\[
2(0.4-1.0)\times0.4\times2=-0.96
\]
负号表示：为了降低误差，`w1` 应该增加。

而第二层权重：
\[
\frac{\partial E}{\partial w_2}
=
2(\hat{y}-target)\cdot h
\]
\[
=2(0.4-1.0)\times1=-1.2
\]
w2 也应该增加。

反向传播的核心不是一个神秘算法，而是：**用链式法则，把最终误差逐层拆回每个参数。**

本书代码把平方误差导数中的常数因子 `2` 吸收到学习率中，因此代码使用 `delta = prediction - target`。只要误差定义和更新方向始终保持一致，训练过程仍然正确。

In [2]:
import numpy as np

x = 2.0
w1 = 0.5
w2 = 0.4
target = 1.0
alpha = 0.1

for step in range(30):
    # 前向传播
    h = x * w1
    prediction = h * w2
    error = (prediction - target) ** 2

    # 反向传播
    delta = prediction - target

    w2_delta = delta * h
    hidden_delta = delta * w2
    w1_delta = hidden_delta * x

    # 同时更新两层
    w1 = w1 - alpha * w1_delta
    w2 = w2 - alpha * w2_delta

    new_h = x * w1
    new_prediction = new_h * w2
    new_error = (new_prediction - target) ** 2

    print(
        step,
        "w1 =", w1,
        "w2 =", w2,
        "prediction =", new_prediction,
        "error =", new_error
    )

0 w1 = 0.548 w2 = 0.46 prediction = 0.50416 error = 0.24585730559999994
1 w1 = 0.59361728 w2 = 0.514344064 prediction = 0.6106470485116519 error = 0.151595720832688
2 w1 = 0.6336695558797824 w2 = 0.5605693920044971 prediction = 0.7104315153425785 error = 0.08384990730679533
3 w1 = 0.6661342017573972 w2 = 0.5972675386184271 prediction = 0.7957206701463827 error = 0.04173004460544298
4 w1 = 0.6905360842638556 w2 = 0.6244830282839422 prediction = 0.8624561300808561 error = 0.018918316152334382
5 w1 = 0.7077148467456555 w2 = 0.6434788293536348 prediction = 0.910799042200163 error = 0.00795681087240831
6 w1 = 0.7191946323261079 w2 = 0.6561045977894102 prediction = 0.9437338099492476 error = 0.003165884142827388
7 w1 = 0.7265779335245862 w2 = 0.6641978661625986 prediction = 0.9651830260957212 error = 0.0012122216718512342
8 w1 = 0.7312030054792784 w2 = 0.6692573151527886 prediction = 0.9787259205574234 error = 0.00045258645612906214
9 w1 = 0.7340505721372956 w2 = 0.672368449318232 prediction

## 6.2 一个输入、两个隐藏节点

现在把隐藏层从一个节点扩展成两个节点：

```text
x (标量)
  ├─ w1[0] → h[0]
  └─ w1[1] → h[1]
h (2,) @ w2 (2,) → prediction (标量)
```

这里必须使用 `np.dot(h, w2)` 或 `h @ w2`。如果误写成 `h * w2`，得到的是两个输出，而不是一个输出。

梯度仍然沿同一条链传播，只是标量变成了长度为 2 的向量：`w2_delta = delta * h`、`hidden_delta = delta * w2`、`w1_delta = hidden_delta * x`。

In [4]:
import numpy as np

x = 2.0
w1 = np.array([0.5, 0.3])
w2 = np.array([0.4, 0.6])

target = 1.0
alpha = 0.1

for step in range(50) :
    h = x * w1
    prediction = np.dot(h, w2)

    delta = prediction - target

    w2_delta = delta * h
    hidden_delta = delta * w2
    w1_delta = hidden_delta * x

    w1 = w1 - w1_delta * alpha
    w2 = w2 - w2_delta * alpha

    new_h = x * w1
    new_prediction = np.dot(new_h, w2)
    new_error = (target - new_prediction) ** 2

    print(
            step,
            "w1 =", w1,
            "w2 =", w2,
            "prediction =", new_prediction,
            "error =", new_error
        )
    

0 w1 = [0.5192 0.3288] w2 = [0.424  0.6144] prediction = 0.84431104 error = 0.024239052265881595
1 w1 = [0.53240242 0.34793106] w2 = [0.44016674 0.62463811] prediction = 0.9033536761588896 error = 0.009340511912000781
2 w1 = [0.54091052 0.36000485] w2 = [0.45045769 0.63136336] prediction = 0.9419023562490474 error = 0.0033753362094126028
3 w1 = [0.54614463 0.367341  ] w2 = [0.45674281 0.63554644] prediction = 0.9658198022574339 error = 0.00116828591772092
4 w1 = [0.54926694 0.37168562] w2 = [0.46047628 0.6380576 ] prediction = 0.9801624672838936 error = 0.00039352770426259346
5 w1 = [0.55109388 0.37421712] w2 = [0.4626555  0.63953227] prediction = 0.98858107721902 error = 0.00013039179747798403
6 w1 = [0.55215049 0.37567767] w2 = [0.46391408 0.6403869 ] prediction = 0.9934588911843674 error = 4.278610453794697e-05
7 w1 = [0.55275739 0.37651544] w2 = [0.46463642 0.64087837] prediction = 0.9962636280161109 error = 1.396047560199135e-05
8 w1 = [0.5531046  0.37699435] w2 = [0.46504948 0.64

## 6.3 多个输入特征：为什么需要 `np.outer`？

现在输入变成 `x.shape == (3,)`，隐藏层仍有两个节点，参数形状为：

```text
x  (3,)
w1 (3, 2)
h  (2,)
w2 (2,)
prediction：标量
```

前向传播是 `h = x @ w1`，再用 `prediction = h @ w2` 得到一个输出。

`w1` 的每一个元素代表一条“输入特征 → 隐藏节点”的连接。因此第一层梯度必须包含所有两两组合：

```python
w1_delta = np.outer(x, hidden_delta)
```

其形状是 `(3, 2)`。矩阵中的第 `i,j` 个元素就是 `x[i] * hidden_delta[j]`。

下面的代码单元沿用前面单元定义的 `target` 和 `alpha`；如果单独运行，请先执行前面的初始化代码。

In [7]:
import numpy as np

x = np.array([2.0, 1.0, -1.0])

w1 = np.array([
    [0.5, 0.1],
    [0.2, 0.3],
    [0.4, 0.2]
])

w2 = np.array([0.4, 0.6])

for step in range(100) :
    h = x @ w1
    prediction = h @ w2

    delta = prediction - target

    w2_delta = delta * h
    hidden_delta = delta * w2
    w1_delta = np.outer(x, hidden_delta)

    w1 = w1 - alpha * w1_delta
    w2 = w2 - alpha * w2_delta

    new_h = x @ w1
    new_prediction = new_h @ w2
    new_error = (new_prediction - target) ** 2

    print("h =", h,
          "w1 =", w1,
          "w2 =", w2,
          "prediction =", prediction,
          "new_prediction =", new_prediction,
          "error =", new_error)

h = [0.8 0.3] w1 = [[0.54 0.16]
 [0.22 0.33]
 [0.38 0.17]] w2 = [0.44  0.615] prediction = 0.5 new_prediction = 0.7 error = 0.09000000000000002
h = [0.92 0.48] w1 = [[0.5664  0.1969 ]
 [0.2332  0.34845]
 [0.3668  0.15155]] w2 = [0.4676 0.6294] prediction = 0.7 new_prediction = 0.8390125 error = 0.025916975156249984
h = [0.9992 0.5907] w1 = [[0.58145555 0.21716511]
 [0.24072778 0.35858255]
 [0.35927222 0.14141745]] w2 = [0.48368587 0.63890953] prediction = 0.8390125 new_prediction = 0.9213919636372845 error = 0.006179223380802005
h = [1.04436665 0.65149532] w1 = [[0.58905987 0.22720979]
 [0.24452994 0.3636049 ]
 [0.35547006 0.1363951 ]] w2 = [0.49189543 0.64403081] prediction = 0.9213919636372845 new_prediction = 0.9639310925166017 error = 0.0013009660870459462
h = [1.06717961 0.68162937] w1 = [[0.5926083  0.23185569]
 [0.24630415 0.36592784]
 [0.35369585 0.13407216]] w2 = [0.49574463 0.64648937] prediction = 0.9639310925166017 new_prediction = 0.9840026188918984 error = 0.0002559162023

## 6.4 多样本同时训练

最后把输入扩展为矩阵：`X.shape == (样本数, 特征数)`。

```text
X       (N, 3)
w1      (3, 2)
H = X @ w1       (N, 2)
w2      (2,)
predictions = H @ w2  (N,)
```

每个样本都有一个误差，但每个参数只有一个更新量，所以要沿样本维度求和或求平均：

```python
w2_delta = H.T @ errors / len(X)
hidden_delta = errors[:, None] * w2
w1_delta = X.T @ hidden_delta / len(X)
```

这就是单样本反向传播的批量版本：先得到每个样本的局部梯度，再把它们汇总到每一条连接。

In [8]:
import numpy as np

X = np.array([
    [2.0, 1.0, -1.0],
    [1.0, 0.0, 2.0],
    [0.0, 3.0, 1.0]
])

targets = np.array([1.1, 0.7, 0.7])

w1 = np.array([
    [0.1, 0.2],
    [0.3, 0.4],
    [0.5, 0.6]
])

w2 = np.array([0.2, 0.3])

alpha = 0.05

for epoch in range(1000):
    # 前向传播
    H = X @ w1
    predictions = H @ w2

    # 每个样本的误差
    errors = predictions - targets

    # 反向传播
    w2_delta = H.T @ errors / len(X)

    hidden_delta = errors[:, None] * w2

    w1_delta = X.T @ hidden_delta / len(X)

    # 更新参数
    w1 -= alpha * w1_delta
    w2 -= alpha * w2_delta

    if epoch % 100 == 0:
        total_error = np.mean(errors ** 2)
        print(
            "epoch =", epoch,
            "predictions =", predictions,
            "error =", total_error
        )

epoch = 0 predictions = [0.06 0.64 0.82] error = 0.3665333333333334
epoch = 100 predictions = [1.02238221 0.70788953 0.76352358] error = 0.0033740036118887705
epoch = 200 predictions = [1.0987297  0.69957361 0.70156224] error = 1.4120193260968053e-06
epoch = 300 predictions = [1.09998631 0.69995429 0.70004741] error = 1.5081747228163526e-09
epoch = 400 predictions = [1.10000022 0.69999677 0.70000218] error = 5.075247223174515e-12
epoch = 500 predictions = [1.10000003 0.69999979 0.70000013] error = 2.0561568662472414e-14
epoch = 600 predictions = [1.1        0.69999999 0.70000001] error = 8.440517497149796e-17
epoch = 700 predictions = [1.1 0.7 0.7] error = 3.4677311804330023e-19
epoch = 800 predictions = [1.1 0.7 0.7] error = 1.4247568270461936e-21
epoch = 900 predictions = [1.1 0.7 0.7] error = 5.8536260600560576e-24


## 第 6 章小结与排错清单

这一章的统一模板是：

```text
前向传播 → 得到预测 → 计算误差 → 链式法则反传 → 更新参数
```

遇到结果不对时，按下面顺序检查：

1. `prediction` 是否应该是标量，是否误用了逐元素乘法 `*`？
2. `w1_delta`、`w2_delta` 是否分别与 `w1`、`w2` 形状一致？
3. 更新后打印的预测值是否重新计算，是否把更新前后的变量混在一起？
4. `delta` 的符号和参数更新式是否保持一致？
5. 多样本训练时，是否沿样本维度汇总梯度？

最终要掌握的不是某一行代码，而是这条判断：**每个参数的梯度，等于误差沿着从该参数到输出的路径逐项相乘；有多个样本时，再把各样本贡献汇总起来。**